# Lab 01 - Formula 1 Driver Performance Analysis: Wet vs. Dry Conditions

In [0]:
CATALOG = 'dbr_dev_ua5816bd'
SCHEMA = 'oles0305'
VOLUME = 'raw_data'

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

### Load CSV files into Delta tables

In [0]:
VOLUME_PATH = f'/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}'
tables_to_load = ('circuits', 'drivers', 'races', 'results')

for table_name in tables_to_load:
    df = (
        spark.read.format('csv')
            .option('inferSchema', True)
            .option('header', True)
            .option('nullValue', r'\N')
            .load(f'{VOLUME_PATH}/{table_name}.csv')
    )
    
    (
        df.write.format('delta')
            .mode('overwrite')
            .option('overwriteSchema', True)
            .saveAsTable(f'{CATALOG}.{SCHEMA}.{table_name}')
    )

In [0]:
from pyspark.sql.functions import col, when, avg, sum, count, round

### Filter target modern races (2022+) and enrich with circuit coordinates

In [0]:
races_df = spark.table(f'{CATALOG}.{SCHEMA}.races')
circuits_df = spark.table(f'{CATALOG}.{SCHEMA}.circuits')

target_races = (
    races_df
        .filter(col('year') >= 2022)
        .join(circuits_df, 'circuitId', how='inner')
        .select(
            col('raceId'),
            col('year'),
            races_df['name'].alias('race_name'),
            col('date'),
            col('lat'),     # Latitude of the circuit
            col('lng')      # Longitude of the circuit
        )
)

target_races.limit(20).display()

races_to_fetch = target_races.collect()
print(f'Races to fetch: {len(races_to_fetch)}')

### Ingest circuit race-day weather (precipitation) via Open-Meteo API

In [0]:
import requests
import builtins

base_url = 'https://archive-api.open-meteo.com/v1/archive'

weather_data = []

for row in races_to_fetch:
    query_params = {
        'latitude': row['lat'],
        'longitude': row['lng'],
        'start_date': str(row['date']),
        'end_date': str(row['date']),
        "hourly": "precipitation"
    }

    try:
        response = requests.get(base_url, params=query_params, timeout=5)
        
        if response.status_code == 200:
            data = response.json()
            # Precipitation between 12:00 and 18:00 (when races are mostly held)
            daytime_precipitation = data['hourly']['precipitation'][12:19]
            
            avg_precipitation = (builtins.sum(daytime_precipitation) / len(daytime_precipitation)) if daytime_precipitation else 0.0
            
            weather_data.append({'raceId': row['raceId'], 'precipitation_mm': float(avg_precipitation)})
    except Exception as e:
        print(f'Exception occured: {e}')
        weather_data.append({'raceId': row['raceId'], 'precipitation_mm': 0.0})

weather_df = spark.createDataFrame(weather_data)
weather_df.limit(10).display()

### Enrich race results with Driver Details, Weather Tags and Positions Gained

In [0]:
drivers_df = spark.table(f'{CATALOG}.{SCHEMA}.drivers')
results_df = spark.table(f'{CATALOG}.{SCHEMA}.results')

race_performance_df = (
    results_df.select('raceId', 'driverId', 'grid', 'position')
        .join(
            drivers_df.select('driverId', 'code', 'forename', 'surname'), 
            on='driverId'
        )
        .join(
            weather_df, 
            on='raceId'
        )
        .filter(col('grid') > 0)
        .withColumn('weather_condition', when(col('precipitation_mm') >= 0.1, 'Wet').otherwise('Dry'))
        .withColumn('positions_gained', col('grid') - col('position'))
        .withColumn('has_finished', when(col('position').isNotNull(), 1).otherwise(0))
)

race_performance_df.limit(10).display()

### Driver performance analytics by track weather condition

In [0]:
driver_weather_analytics_df = (
    race_performance_df
        .withColumnRenamed('code', 'driver_code')
        .groupBy('driver_code', 'forename', 'surname', 'weather_condition')
        .agg(
            round(avg(col('positions_gained')), 2).alias('avg_positions_gained'),
            round(sum(col('has_finished')) / count(col('raceId')) * 100, 2).alias('finishes_percentage'),
            count(col('raceId')).alias('races_count')
        )
        .filter(col('races_count') >= 5)
        .orderBy(col('driver_code'), col('weather_condition'))
)

driver_weather_analytics_df.display()

### Save the analytics to Delta Table

In [0]:
(
    driver_weather_analytics_df.write.format('delta')
        .mode('overwrite')
        .option('overwriteSchema', True)
        .saveAsTable(f'{CATALOG}.{SCHEMA}.f1_driver_weather_analytics')
)

### SQL View: Wet weather driver performance

In [0]:
%sql
CREATE OR REPLACE VIEW dbr_dev_ua5816bd.oles0305.driver_groups_view AS
    SELECT surname,
        driver_code,
        avg_positions_gained,
        finishes_percentage,
    CASE 
        WHEN avg_positions_gained > 0 AND finishes_percentage >= 90 THEN 'Rainmaster (High Gain, Safe)'
        WHEN avg_positions_gained > 0 AND finishes_percentage < 90  THEN 'Gambler (High Gain, High Risk)'
        WHEN avg_positions_gained <= 0  AND finishes_percentage >= 90 THEN 'Cruiser (Low Gain, Safe)'
        ELSE 'Struggler (Low Gain, High Risk)'
    END AS driver_category
    FROM dbr_dev_ua5816bd.oles0305.f1_driver_weather_analytics
    WHERE weather_condition = 'Wet'
